# PLS_Regression — previsões e resíduos

**Lê:** `predictions_PLS_Regression.csv`

**Grava:** `outputs/figures/previsao_PLS_Regression.png` e `residuos_PLS_Regression.png`

In [ ]:
from pathlib import Path
import sys
from IPython.display import display, Image, Markdown
import pandas as pd

# sobe até achar analyses/grupo2
here = Path.cwd().resolve()
group = None
for c in [here, *here.parents]:
    if (c / 'config.py').exists() and (c / 'lib').is_dir():
        group = c
        break
if group is None:
    raise RuntimeError('Abra/rode este notebook de dentro de analyses/grupo2/...')
if str(group) not in sys.path:
    sys.path.insert(0, str(group))

import config
from lib import data as data_lib
from lib import features, eda_stl, models
from lib.nb_boot import require_file, save_json, load_json
data_lib.ensure_output_dirs()
print('grupo2 →', group)
print('outputs →', config.OUTPUT_DIR)


In [ ]:
import matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
pred = pd.read_csv(require_file(config.RESULT_DIR / 'predictions_PLS_Regression.csv',
                               '02_modelos/PLS_Regression/02_walkforward_validacao.ipynb'),
                  parse_dates=['date_time'])
resid = pred['y_true'] - pred['y_pred']
print('MAE', resid.abs().mean())

fig, ax = plt.subplots(figsize=(11, 4))
s = pred.iloc[:24*14]
ax.plot(s['date_time'], s['y_true'], label='Real')
ax.plot(s['date_time'], s['y_pred'], label='Previsto')
ax.set_title('Previsão vs real (14 dias) — PLS_Regression')
ax.legend(); fig.tight_layout()
fig.savefig(config.FIG_DIR / 'previsao_PLS_Regression.png', dpi=120); plt.show()

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
axes[0].plot(pred['date_time'], resid, lw=0.5)
axes[0].axhline(0, color='red', lw=0.8)
axes[0].set_title('Resíduos — PLS_Regression')
plot_acf(resid, lags=48, ax=axes[1])
fig.tight_layout()
fig.savefig(config.FIG_DIR / 'residuos_PLS_Regression.png', dpi=120); plt.show()
